In [31]:
import pandas as pd
import numpy as np
import glicko2
import copy

#from fighters_fights.csv create a dataframe with fighter, date, glicko rating

folder = '/Users/alejandrogomez-paz/Desktop/UFC Project/2. data_cleaning/'
df = pd.read_csv(folder + 'fighters_fights.csv')
df = df.sort_values('date', ascending=True).reset_index(drop=True)
rating_dict = {} # [fighter, date]: glicko rating 
initialized_fighters = set()

for i, row in df.iterrows():
    fighter, opponent, date = row['fighter_name'], row['opponent_name'], row['date']
    if fighter not in initialized_fighters:
        rating_dict[(fighter, date)] = glicko2.Player()
        initialized_fighters.add(fighter)
    if opponent not in initialized_fighters:
        rating_dict[(opponent, date)] = glicko2.Player()
        initialized_fighters.add(opponent)

    last_date = max([d for (f, d) in rating_dict.keys() if f == fighter])
    opp_last_date = max([d for (f, d) in rating_dict.keys() if f == opponent])
    fighter_object = copy.deepcopy(rating_dict[(fighter, last_date)])
    opponent_object = copy.deepcopy(rating_dict[(opponent, opp_last_date)])

    if fighter == row['winner_name']:
        fighter_object.update_player([rating_dict[(opponent, opp_last_date)].rating],
                                                                                  [rating_dict[(opponent, opp_last_date)].rd], [1])
        rating_dict[(fighter, date)] = fighter_object
    else:
        fighter_object.update_player([rating_dict[(opponent, opp_last_date)].rating],
                                                                                  [rating_dict[(opponent, opp_last_date)].rd], [0])
        rating_dict[(fighter, date)] = fighter_object

df_glicko = pd.DataFrame(
    [(f, d, p.rating, p.rd, p.vol) for (f, d), p in rating_dict.items()],
    columns=['fighter', 'date', 'rating', 'rating_deviation', 'volatility'])

df_glicko.to_csv('glicko.csv', index = False)